<table style="width:100%; border-bottom: 2px solid #ccc; margin-bottom: 20px;">
  <tr>
    <td style="vertical-align:middle;">
      <img src="../resources/ADI-Logo-RGB-FullColor.png" alt="Company Logo" height="30">
    </td>
    <td style="text-align:right; vertical-align:middle;">
      <p style="margin: 0;">Phased Array Systems</p>
      <p style="font-size: 14px; margin: 0;">Iain Derrington - ADEF Group, ADI</p>
      <p style="font-size: 12px; color: #555;">Field Applications & Platform Engineer</p>
    </td>
  </tr>
</table>

In [ ]:
# Common Declarations and setup
import os
import sys
import time

sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from phaser_functions import *
from phaser_init import init_phaser_sdr

from adi import adf4159
from adi import ad9361
from adi import one_bit_adc_dac
from adi import ad9361
from adi import tddn
from adi.cn0566 import CN0566

import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output 

from dataclasses import dataclass, fields
from typing import List

# Get Script / Notebook root and full path to resources folder
phaser_root = get_phaser_root()
resource_path = phaser_root / "resources"

display(Markdown(f"Phaser root: **{phaser_root}**"))
display(Markdown(f"Resource path: **{resource_path}**\n"))


# FMCW RADAR: Synchronized Range Measurement with TDD

## Overview

In the previous notebook, we demonstrated FMCW chirp generation but encountered the **timing synchronization problem**:
- Random capture timing meant we couldn't reliably determine if we were sampling an up-chirp or down-chirp
- TX-RX coupling dominated the received signal, masking target returns
- Without synchronization, range measurement was impossible

</details>
<table style="width:100%; border-bottom: 2px solid #ccc; margin-bottom: 20px;">
  <tr>
    <td style="vertical-align:middle;"> <img src="resources/pluto-normal.svg" alt="SA Chirp Frequency" width="600"></td>
  </tr>
</table>

This notebook solves these problems using **TDD (Time Division Duplex)** engine with synchronized triggering.

## Learning Objectives

By the end of this notebook, you will:
1. Understand TDD operation and why it's essential for FMCW RADAR
2. Configure synchronized chirp generation and data capture using `sync_start`
3. Process the Range FFT to convert beat frequency to target range
4. Perform range calibration and measure static targets accurately
5. Understand practical limitations (minimum/maximum range, range resolution)

## Section 1: Using TDD for Timing

### What is TDD?

Time Division Duplexing (TDD) is normally used to separate transmit and receive activity in time:

- **TX period**: transmit is active, receive is disabled or ignored
- **RX period**: receive is active, transmit is disabled or reduced

In this FMCW radar example, however, transmit and receive occur at the same time. We are not using the TDD engine to alternate between TX and RX. Instead, we use it as a hardware timing controller to synchronize:

1. ADF4159 chirp generation
2. TX buffer playback
3. RX buffer capture

In the previous notebook, these events were not synchronized. That meant each capture could begin at a different point in the chirp, making the beat frequency difficult to interpret reliably. In this notebook, we fix that by using the PlutoSDR TDD engine and the ADF4159 TXDATA trigger input.

### The Aim

The aim of this notebook is to tidy up what we did in the previous exercises.

In the previous notebook, we were able to generate, transmit, and receive linear FMCW chirps. However, we could not guarantee the timing relationship between chirp generation, transmit playback, and receive capture. That makes range measurement unreliable, because each RX buffer may start at a different point in the chirp.

In this notebook, we will use the PlutoSDR TDD engine to align the key timing events:

- **Chirp generation**: trigger the ADF4159 ramp at a known point in the TDD frame
- **RX buffer**: capture samples at a known time relative to the chirp
- **TX buffer**: play the transmit baseband signal at the required time

### How?

We will use two features:

1. The PlutoSDR TDD engine, controlled through IIO
2. The ADF4159 `TXDATA` pin, which can be configured to start a ramp from an external trigger

The TDD engine is exposed as one of the IIO devices on the PlutoSDR. Let's inspect the available IIO devices using `iio_attr`:

In [ ]:
! iio_attr -u "ip:192.168.2.1" -d 

We can see one of the devices is assocaited with TDD:

`iio:device5, iio-axi-tdd-0: found 18 device attributes`

iio-axi-tdd-0 is a IIO represetation of ADI's [TDD](https://analogdevicesinc.github.io/hdl/library/axi_tdd/index.html#axi-tdd) engine.

Lets have a look at what channels and attributes we have:

In [ ]:
! iio_attr -u "ip:192.168.2.1" -c iio-axi-tdd-0
! iio_attr -u "ip:192.168.2.1" -d iio-axi-tdd-0 

In [ ]:
! iio_attr -u "ip:192.168.2.1" -c iio-axi-tdd-0 channel0

What can we conclude from this?

The TDD engine exposes three output channels. Each channel can generate a timed control signal within the TDD frame.

This is useful for our FMCW setup because we need to coordinate three timing-related events:

1. Trigger the ADF4159 chirp ramp
2. Start or align RX buffer capture
3. Start or align TX buffer playback

In other words, the TDD engine gives us a shared timing reference for the chirp generator, receiver, and transmitter.

<details>
<summary>Deep dive into the TDD attributes</summary>

Lets look into the device TDD device attributes first:

#### AXI TDD Device Attributes

| IIO Attribute              | Register (logical name)  | Description                                                                                                                                             |
| -------------------------- | ------------------------ | ------------------------------------------------------------------------------------------------------------------------------------------------------- |
| `enable`                   | `ENABLE`                 | Master enable for the TDD engine (starts/stops the counter and FSM).  |
| `frame_length_raw`         | `FRAME_LENGTH`           | Length of one frame in **clock cycles** (sets the counter wrap point). |
| `frame_length_ms`          | `FRAME_LENGTH` (scaled)  | Same as above but expressed in milliseconds (converted in driver)                                                                                       |
| `burst_count`              | `BURST_COUNT`            | Number of frames to run; `0 = continuous operation`.                   |
| `startup_delay_raw`        | `STARTUP_DELAY`          | Delay (in clock cycles) before the first frame starts after enable                                                                                      |
| `startup_delay_ms`         | `STARTUP_DELAY` (scaled) | Same delay expressed in milliseconds                                                                                                                    |
| `internal_sync_period_raw` | `SYNC_PERIOD`            | Period of internal sync generator (in cycles)                                                                                                           |
| `internal_sync_period_ms`  | `SYNC_PERIOD` (scaled)   | Same period expressed in milliseconds                                                                                                                   |
| `sync_internal`            | `SYNC_CONTROL`           | Enables internal periodic sync source                                                                                                                   |
| `sync_external`            | `SYNC_CONTROL`           | Enables external sync input                                                                                                                             |
| `sync_soft`                | `SYNC_CONTROL` (trigger) | Software-triggered sync pulse (write triggers event)                                                                                                    |
| `sync_reset`               | `SYNC_CONTROL`           | Resets/re-arms sync logic                                                                                                                               |
| `state`                    | `STATE`                  | FSM state of the engine (e.g. idle / waiting / running)                                                                                                 |
| `waiting_for_supplier`     | `STATUS`                 | Indicates the engine is waiting for a sync event                                                                                                        |
                                                                                                             
The device attributes are used to setup the main counter and any sycronisation demands.
They define:
* how the **counter runs**
* when it **starts**
* how it **repeats**
* All actual waveform behaviour comes later in the **channel attrbutes**

The channel attributes are:

#### AXI TDD - Channel Attributes (per channel)

| IIO Attribute | Register (logical name) | Description                                                                                  |
| ------------- | ----------------------- | -------------------------------------------------------------------------------------------- |
| `enable`      | `CHANNEL_ENABLE`        | Enables/disables this channel output. When `0`, the output is inactive regardless of timing. |
| `polarity`    | `CHANNEL_POLARITY`      | Defines active level: `0 = active HIGH`, `1 = active LOW`.                                   |
| `on_raw`      | `CHANNEL_ON`            | Start time of the active window (in clock cycles from frame start).                          |
| `on_ms`       | `CHANNEL_ON` (scaled)   | Same start time expressed in milliseconds.                                                   |
| `off_raw`     | `CHANNEL_OFF`           | End time of the active window (in clock cycles from frame start).                            |
| `off_ms`      | `CHANNEL_OFF` (scaled)  | Same end time expressed in milliseconds.                                                     |

***

Each channel implements:

```
IF (counter >= on) AND (counter < off)
    -> output = active (respecting polarity)
ELSE
    -> output = inactive
```

All channels use the **same counter**
So they are **perfectly aligned in time**

Content here.

</details>

## Section 2: Hardware Configuration

The Phaser kit is already configured to support TDD, so this section is only a brief overview of the required hardware connections.

Both the PlutoSDR and the ADF4159 need to be configured to work with the TDD engine. The TDD engine is implemented inside the Zynq programmable logic used by the PlutoSDR, so the timing signals used to synchronize the TX and RX buffers are internal to the Pluto design.

The ADF4159, however, is external to the PlutoSDR. To trigger the chirp ramp from the TDD engine, a TDD output signal must be routed to the ADF4159 `TXDATA` pin.

### ADF4159 Hardware Configuration for Sync

The ADF4159 `TXDATA` pin is routed to pin 1 of P1, the dual-row 0.1 inch header on the Phaser main board. This signal is connected to the PlutoSDR Zynq through `PL_GPIO0`.

| Phaser Signal | Pluto PL GPIO       | Zynq Package Pin |
| ------------- | ------------------- | ---------------- |
| TXDATA        | PL_GPIO0 (L10P)     | K13              |
| BURST         | PL_GPIO1            | M12              |
| MUXOUT        | PL_GPIO2            | R10              |
| I2C SDA       | PL_GPIO3            | N14              |
| I2C SCL       | PL_GPIO4            | M14              |

When the ADF4159 is configured for `TX

## Section 3: Software Configuration

Now that we have covered the timing architecture and hardware connections, we can configure the system in software.

The setup sequence is:

1. Create the IIO device objects
2. Configure the Phaser board
3. Configure the PlutoSDR
4. Configure the ADF4159 for externally triggered chirps
5. Configure the TDD engine
6. Load the TX baseband waveform

This order matters. The SDR and Phaser hardware must be initialized before we configure the chirp source and timing engine, and the TDD engine should be configured before we begin triggering captures.

Let's start with the common imports and configuration parameters:

In [ ]:
@dataclass
class RadarConfig:
    """Configuration parameters for FMCW radar operation."""

    # ========== SDR Parameters ==========
    sample_rate: float = 5.85e6  # NOTE: This is a placeholder! The actual sample rate is calculated
                                  # in configure_sdr() based on buffer_size / frame_time to ensure
                                  # we capture exactly one complete frame (chirp + padding).
                                  # Formula: sample_rate = sdr_buf_size / (ramp_time + pri_padding_ms)
                                  # For 4096 samples / 0.7ms = 5.85 MHz
                                  # ALWAYS use sdr.sample_rate (actual) not config.sample_rate (default) in plots!

    center_freq: float = 2.1e9  # SDR LO frequency (Hz). Upconverted to output_freq by ADF4159
                                 # Keep at 2.1 GHz for optimal Pluto performance

    signal_freq: float = 100e3  # TX baseband tone frequency (Hz). Creates IF offset
                                 # Used to separate DC offset from target returns
                                 # Typical: 100 kHz. Don't change unless you know why

    rx_gain: int = 60  # Receiver gain (dB). Range: -3 to 70 dB
                        # Higher = more sensitive but risk ADC saturation on strong returns
                        # Start at 60, reduce if seeing saturation artifacts
                        # Trade-off: +10 dB gain ~= 3x detection range OR 10 dB less TX power needed

    tx_gain: int = 0  # Transmitter gain (dB). Range: 0 to -88 dB (0 = max power)
                       # 0 dB ~= +30 dBm EIRP with array gain ~= 1W effective
                       # Reduce for short range, regulations, or power saving
                       # Trade-off: -6 dB power ~= 0.5x detection range

    sdr_buf_size: int = 1024 * 4
    fft_size: int = sdr_buf_size

    # ========== Chirp Parameters ==========
    output_freq: float = 9.9e9  # Radar transmit frequency (Hz). X-band (8-12 GHz)
                                 # 9.9 GHz = 30mm wavelength. Good for small targets
                                 # Check local regulations (ISM, amateur, Part 15)

    chirp_BW: float = 500e6  # Chirp bandwidth (Hz). Determines range resolution
                              # Range resolution = c/(2*BW) = 0.3m at 500 MHz
                              # Wider BW = better resolution, more processing
                              # Typical: 250 MHz to 1 GHz (if hardware supports)
                              # Trade-off: 2x BW = 0.5x range resolution (better)

    ramp_time: int = 500  # Chirp duration (microseconds). Affects max range
                           # Longer = more samples per chirp = better range resolution
                           # Also affects PRF (pulse repetition frequency)
                           # Typical: 100 to 1000 us
                           # Trade-off: 2x ramp_time = 0.5x PRF = 0.5x max unambiguous velocity

    num_chirps: int = 1  # Number of chirps per frame (CPI - Coherent Processing Interval)
                            # More chirps = better Doppler (velocity) resolution
                            # Doppler resolution = lambda/(2*CPI*PRI) where PRI ~= ramp_time
                            # Typical: 64 to 512. Power of 2 for efficient FFT
                            # Trade-off: 2x chirps = 0.5x Doppler resolution, 2x processing time

    # ========== Array Parameters ==========
    element_spacing: float = 0.014  # Antenna element spacing (meters). 14mm ~= lambda/2 at 10 GHz
                                     # lambda/2 spacing prevents grating lobes (spatial aliasing)
                                     # Don't change unless physical array changes

    gain_list: List[int] = None  # Per-element gain (0-127). None = all max (127)
                                  # Can apply taper (Blackman, Taylor) to reduce sidelobes
                                  # Example: [8, 34, 84, 127, 127, 84, 34, 8] for Blackman
                                  # Trade-off: Tapering reduces sidelobes but lowers gain

    # ========== Timing Parameters (Advanced) ==========
    begin_offset_fraction: float = 0.1  # Fraction of chirp to skip at start (0.0-0.3)
                                         # VCO takes time to settle; early samples are non-linear
                                         # 0.1 = skip first 10% of chirp (30 us at 300 us ramp)
                                         # Increase if seeing range artifacts near zero
                                         # Trade-off: More offset = fewer samples = less SNR

    pri_padding_ms: float = 0.1   # Dead time between chirps (milliseconds)
                                  # Allows VCO to reset and prevents chirp overlap
                                  # PRI (Pulse Repetition Interval) = ramp_time + padding
                                  # Affects PRF and max unambiguous velocity
                                  # Trade-off: More padding = lower PRF = lower max velocity

    # ========== TDD (Time Division Duplex) Parameters ==========
    tdd_trigger_on_raw: int = 0   # TDD GPIO trigger start (raw units)
                                   # Synchronizes chirp generation with data capture
                                   # Keep at 0 for immediate trigger

    tdd_trigger_off_raw: int = 20  # TDD GPIO trigger stop (raw units)
                                    # Pulse width for trigger signal
                                    # Typical: 5-20. Must be long enough for hardware to latch
                                    # Trade-off: Longer pulse more reliable but delays start

    #rpi_ip:       str = "192.168.1.10"
    rpi_ip:       str = "phaser.local"
    sdr_ip:       str = "192.168.2.1"
    fieldfox_ip:  str = "192.168.1.30"

    def __post_init__(self):
        """Set default gain list and calibration file paths if not provided."""
        if self.gain_list is None:
            self.gain_list = [127] * 8

    def __iter__(self):
        for field in fields(self):
            yield field.name, getattr(self, field.name)

#  Default are stored in a dataclass
config = RadarConfig()

display(Markdown("#### Config values"))
for name, value in config:    
    display(Markdown(f"{name} = {value}"))

pll    = None      
gpio   = None
tdd    = None
phaser = None
sdr    = None


### IIO Connection

First, we create the IIO device objects used throughout the notebook.

The Phaser system uses two network endpoints:

- **Raspberry Pi / Phaser controller**: used for the CN0566 board, ADF4159 PLL, and GPIO control
- **PlutoSDR**: used for the AD9361 SDR and the TDD engine

Creating these objects gives Python access to the hardware attributes exposed through IIO.

In [ ]:
def connect_devices():
    try:
        global pll, gpio, tdd, phaser, sdr
        display(Markdown(f"- ADF4159: ip: {config.rpi_ip}"))
        pll    = adf4159        (uri="ip:" + config.rpi_ip)
        
        display(Markdown(f"- GPIO: ip: {config.rpi_ip}"))
        gpio   = one_bit_adc_dac(uri="ip:" + config.rpi_ip)

        display(Markdown(f"- TDDN: ip: {config.sdr_ip}"))
        tdd = tddn(uri="ip:" + config.sdr_ip)

        display(Markdown(f"- Phaser: ip: {config.rpi_ip}"))
        phaser = CN0566         (uri="ip:" + config.rpi_ip)

        display(Markdown(f"- SDR: ip: {config.rpi_ip}"))
        sdr    = ad9361         (uri="ip:" + config.sdr_ip)
    
    except:
        display(Markdown(f"Unable to connect to CN0566 / ADF4159. Please check the IP addresses and connections."))
        print("")
        sys.exit(1)
    
    # Set phaser.sdr to instance of PlutoSDR
    phaser.sdr = sdr 


### Configuration of the Phaser Board

The Phaser board configuration is similar to the previous notebook, so we will only summarize the key settings here.

In this setup we:

- configure the CN0566 for receive operation
- set the antenna element spacing
- set all channel phases to 0 degrees
- apply the configured per-channel gains
- enable the required RF signal path using the Phaser GPIO controls

These settings provide a known starting point before we configure the SDR, chirp generator, and TDD timing.

In [ ]:
def configure_phaser():
    phaser.configure(device_mode="rx")
    phaser.element_spacing = config.element_spacing
    
    for i in range(0, 8):
        phaser.set_chan_phase(i, 0)

    display(Markdown(f"- Phaser channel phase  = 0"))
    
    for i in range(0, len(config.gain_list)):
        phaser.set_chan_gain(i, config.gain_list[i], apply_cal=False)
    
    phaser._gpios.gpio_tx_sw = 0
    phaser._gpios.gpio_vctrl_1 = 1
    phaser._gpios.gpio_vctrl_2 = 1


### PlutoSDR Configuration

Next, we configure the PlutoSDR transmit and receive paths.

The receive buffer size and sample rate are chosen so that one RX buffer captures one complete TDD frame: the chirp duration plus the padding time before the next chirp. This gives each capture a fixed timing relationship to the chirp trigger.

In this setup we configure:

- RX and TX local oscillator frequencies
- RX and TX enabled channels
- RX and TX buffer sizes
- manual RX gain
- TX output power
- cyclic TX buffer mode for continuous baseband tone playback

The actual sample rate is calculated from the desired frame time and buffer size, so later processing should use `sdr.sample_rate` rather than the placeholder value in `config.sample_rate`.

In [ ]:
def configure_sdr():   
    destroy_sdr_buffer()
    
    # Configure sample rate to capture the full frame (chirp + padding)
    # Frame time = ramp_time + pri_padding_ms
    frame_time_s = (config.ramp_time * 1e-6) + (config.pri_padding_ms * 1e-3)
    phaser.sdr.sample_rate = int(config.sdr_buf_size / frame_time_s)
    
    phaser.sdr.rx_lo = int(config.center_freq)
    phaser.sdr.rx_enabled_channels = [0, 1]
    phaser.sdr.rx_buffer_size = config.sdr_buf_size
    
    phaser.sdr.gain_control_mode_chan0 = 'manual'
    phaser.sdr.gain_control_mode_chan1 = 'manual'
    phaser.sdr.rx_hardwaregain_chan0 = -88
    phaser.sdr.rx_hardwaregain_chan1 = config.rx_gain

    phaser.sdr.tx_buffer_size = config.sdr_buf_size
    phaser.sdr.tx_lo = int(config.center_freq)
    phaser.sdr.tx_enabled_channels = [0, 1]
    phaser.sdr.tx_cyclic_buffer = True
    phaser.sdr.tx_hardwaregain_chan0 = -88
    phaser.sdr.tx_hardwaregain_chan1 = int(config.tx_gain)

    display(Markdown(f""""- Sample rate: {sdr.sample_rate/1e6:.2f} MHz
- Frame time: {frame_time_s*1e3:.2f} ms (chirp + padding)
- Chirp time: {config.ramp_time} us
- Padding time: {config.pri_padding_ms} ms
- RX LO: {sdr.rx_lo/1e9:.1f} GHz
- TX LO: {sdr.tx_lo/1e9:.1f} GHz
- Buffer size: {sdr.rx_buffer_size} samples
- Capture time: {sdr.rx_buffer_size/sdr.sample_rate*1e3:.2f} ms"""))

def destroy_sdr_buffer():
    try: sdr.tx_destroy_buffer()
    except: pass
    try: sdr.rx_destroy_buffer()
    except: pass

### ADF4159 Configuration for Triggered Chirps

Next, we configure the ADF4159 to generate a chirp only when it receives an external trigger.

This is different from the continuous chirp mode used previously. In triggered mode, the PLL waits for a rising edge on the `TXDATA` pin before starting the ramp. That allows the TDD engine to control exactly when each chirp begins.

Key settings are:

- `ramp_mode = "single_sawtooth_burst"`: generate one sawtooth ramp per trigger
- `tx_trig_en = 1`: enable external triggering from the `TXDATA` pin
- chirp bandwidth, ramp time, and step size: define the FMCW slope used for range measurement

In [ ]:
def configure_adf4159():
    # Configure ADF4159 for triggered sawtooth chirp

    vco_freq = int(config.output_freq + config.signal_freq + config.center_freq)
    BW = config.chirp_BW
    num_steps = int(config.ramp_time)

    display(Markdown(f"- VCO Output Frequency = {vco_freq/1e9} GHz"))

    phaser.frequency = int(vco_freq / 4)
    phaser.freq_dev_range = int(BW / 4)
    phaser.freq_dev_step = int((BW / 4) / num_steps)
    phaser.freq_dev_time = int(config.ramp_time)
    
    phaser.delay_word = 4095
    phaser.delay_clk = "PFD"
    phaser.delay_start_en = 0
    phaser.ramp_delay_en = 0
    phaser.trig_delay_en = 0
    phaser.ramp_mode = "single_sawtooth_burst"
    phaser.sing_ful_tri = 0
    phaser.tx_trig_en = 1
    phaser.enable = 0                                  # 0 = PLL enable.  Write this last to update all the registers

    display(Markdown(f"- Chirp BW: {(4 * phaser.freq_dev_range)/1e6:.0f} MHz"))
    display(Markdown(f"- Ramp time: {config.ramp_time} us"))
    display(Markdown(f"- Chirp rate: {config.chirp_BW/(config.ramp_time*1e-6)/1e12:.2f} THz/s"))


### TDD Configuration

The PlutoSDR TDD engine provides hardware-timed control signals for synchronizing the FMCW measurement. This avoids relying on software timing, which can vary because of operating system and network latency.

In this notebook, the TDD sequence is started by a hardware trigger pulse. Once triggered, the TDD engine generates a repeatable timing frame for each chirp.

Within each frame, the TDD channels can be used to align:

- the ADF4159 chirp trigger
- the receiver capture window
- the transmitter enable window

Each TDD channel has an `on` time and an `off` time, referenced to the start of the TDD frame. The frame length sets the time between successive chirps, and the burst count sets how many chirps are generated from one hardware trigger event.

In the configuration below:

- `sync_external = True` selects the hardware trigger input
- `frame_length_ms` is set to the chirp duration plus padding time
- `burst_count` is set to the number of chirps per burst
- channel 0 generates the trigger pulse for the ADF4159 `TXDATA` input
- channels 1 and 2 provide synchronized RX and TX timing signals

Later, the trigger pulse is generated by toggling the Phaser `BURST` GPIO line, which is routed to the PlutoSDR TDD external sync input.

In [ ]:
def configure_tdd():
    """
    Configure the TDD (Time Division Duplex) controller so that each FMCW chirp
    is synchronised to the data acquisition hardware.

    The TDD engine generates trigger signals at the start of each chirp period
    (PRI) and repeats this for the required number of chirps in a burst.
    """

    # Route the TDD trigger to the external sync circuitry and enable
    # the Phaser board trigger path.
    gpio.gpio_tdd_ext_sync = True
    gpio.gpio_phaser_enable = True

    # Disable the TDD engine while its configuration is updated.
    tdd.enable = False

    # Use an external trigger source to start the TDD sequence.
    tdd.sync_external = True

    # Begin generating triggers immediately after synchronisation.
    tdd.startup_delay_ms = 0

    # Calculate the Pulse Repetition Interval (PRI).
    #
    # The PRI consists of:
    #   - the FMCW ramp (chirp) duration
    #   - additional padding time between chirps
    #
    # ramp_time is stored in microseconds, so convert to milliseconds
    # before adding the padding value.
    PRI_ms = config.ramp_time / 1e3 + config.pri_padding_ms

    
    # Set the time between successive chirp triggers.
    tdd.frame_length_ms = PRI_ms
    # Generate one trigger event per chirp in the burst.
    tdd.burst_count = config.num_chirps

    tdd.channel[0].enable = True
    tdd.channel[0].polarity = False
    tdd.channel[0].on_raw = config.tdd_trigger_on_raw
    tdd.channel[0].off_raw = config.tdd_trigger_off_raw

    tdd.channel[1].enable = True
    tdd.channel[1].polarity = False
    tdd.channel[1].on_raw = config.tdd_trigger_on_raw
    tdd.channel[1].off_raw = config.tdd_trigger_off_raw
    
    tdd.channel[2].enable = True
    tdd.channel[2].polarity = False
    tdd.channel[2].on_raw = config.tdd_trigger_on_raw
    tdd.channel[2].off_raw = config.tdd_trigger_off_raw
    
    # Apply the configuration and start the TDD engine.
    tdd.enable = True

    display(Markdown(f""""- Frame Len =  {tdd.frame_length_ms} ms
- No. Chirps / trigger =  {config.num_chirps}"""))


### TX Baseband Signal

Next, we generate the PlutoSDR transmit baseband signal.

The ADF4159 controls the RF chirp by sweeping the Phaser LO, while the PlutoSDR provides a fixed complex baseband tone. In this example, the tone is set by `config.signal_freq`, typically 100 kHz.

Using a non-zero baseband tone shifts the received signal away from DC. This helps separate the target beat signal from DC offset, LO leakage, and other low-frequency artifacts.

The generated IQ waveform is loaded into the PlutoSDR TX buffer and transmitted cyclically, so the baseband tone is already present whenever the TDD timing enables the transmit path.

In [ ]:
def tx_baseband():
    # Generate baseband transmit waveform (tone at IF)

    fs = int(sdr.sample_rate)
    N = config.sdr_buf_size
    t = np.arange(N) / fs  # FIXED: Guarantees exactly N samples
    
    i_data = np.cos(2 * np.pi * t * config.signal_freq) * 2**14
    q_data = np.sin(2 * np.pi * t * config.signal_freq) * 2**14
    iq_data = i_data + 1j * q_data

    # Validate buffer length BEFORE sending to SDR
    expected_tx_buffer = sdr.tx_buffer_size
    actual_samples = len(iq_data)
    
    display(Markdown(f"- Transmit {config.signal_freq/1e3} kHz"))
    display(Markdown(f"- TX buffer size: {expected_tx_buffer} (expected) vs {actual_samples} (actual)"))
    
    if actual_samples != expected_tx_buffer:
        raise ValueError(f"❌ Buffer length mismatch! Expected {expected_tx_buffer}, got {actual_samples}")
    
    # Send waveform to TX buffer
    sdr.tx([iq_data, iq_data])
    
    display(Markdown(f"✓ TX buffer loaded successfully"))

### Run Configuration

Now we run the configuration sequence.

This cell connects to the hardware, configures the Phaser board and PlutoSDR, programs the ADF4159 for triggered chirps, enables the TDD engine, and loads the TX baseband waveform.

After this cell completes, the system is ready to generate hardware-triggered, synchronized FMCW captures.

In [ ]:
display(Markdown("**Connect to Devices**"))
connect_devices()

display(Markdown("**Configure Phaser**"))
configure_phaser()

display(Markdown("**Configuring PlutoSDR**"))
destroy_sdr_buffer()
configure_sdr()

display(Markdown("**Configuring ADF4159**"))
configure_adf4159()    

display(Markdown("**Configure TDD Engine**"))
configure_tdd()      

display(Markdown("**Transmit Baseband Signal**"))
tx_baseband()

## Configuration Summary

At this point, the FMCW RADAR system is fully configured for synchronized range measurement:

### Hardware Configuration
- **Phaser Board**: Configured in RX mode with all 8 channels phase-aligned
- **ADF4159 PLL**: 
  - Chirp bandwidth: **500 MHz** (10.0 - 10.5 GHz at RF)
  - Chirp duration: **500 us** (sawtooth ramp)
  - Chirp rate: **1.0 THz/s**
  - Trigger mode: **Single sawtooth burst** (triggered by TDD TXDATA)
- **PlutoSDR**:
  - RX buffer: **4096 samples** (~1 chirp)
  - Sample rate: **4096 / 600e-6 = 6.83 MHz  (1 Buffer = 1 Chirp**
  - RX LO: **2.1 GHz** (downconverts to 100 kHz IF)
  - TX LO: **2.1 GHz** (upconverts 100 kHz baseband tone)

### TDD Synchronization
- **TDD Engine**: Enabled and configured
- **Trigger mode**: Hardware trigger (`tdd.sync_external = True`)
- **Frame period**: 1.5 ms (500 us chirp + 100 us padding)
- **Burst count**: **1 chirp** per trigger
- **Channel routing**: 
  - Channel[0] -> ADF4159 TXDATA pin (starts chirp)
  - Channel[1] -> PlutoSDR RX enable (synchronized capture)
  - Channel[2] -> PlutoSDR TX enable (synchronized transmit)

### Signal Flow
1. **TDD trigger** (hardware trigger) starts the sequence
2. **ADF4159** begins frequency ramp from 10.0 -> 10.5 GHz
3. **TX** transmits 100 kHz IF tone (upconverted to RF)
4. **Target reflection** returns with time delay tau
5. **RX** captures reflected signal (downconverted to 100 kHz IF)
6. **De-chirping** (mixing TX and RX) produces a beat frequency: f_beat is proportional to range
7. **FFT** converts beat frequency to range bins

### Expected Range Performance
- **Range resolution**: 30 cm (c / 2B = 3e8 / (2 * 500e6))
- **Maximum unambiguous range**: 45 m (at 600 kHz sample rate)
- **Current target**: Brick wall at ~2 m

The system is now ready to capture synchronized chirps and measure range!

## Live Beat Frequency Display

Now let's capture the de-chirped signal and visualize the beat frequency in real-time. 

The beat frequency appears at:
$$
f_{beat} = \frac{2 \cdot R \cdot B}{c \cdot T}
$$

For a 2m target with our 500 MHz chirp:
$$
f_{beat} = \frac{2 \times 2 \times 500 \times 10^6}{3 \times 10^8 \times 500 \times 10^{-6}} = 13.3 \text{ kHz}
$$

**Important**: In a properly synchronized FMCW system, the beat frequency should appear at **only one frequency** (positive), not both positive and negative. If you see mirror images, it indicates timing/synchronization issues.

In [ ]:
def capture_data():
    """
    Trigger TDD and capture synchronized data
    """
    # Trigger TDD sequence (starts one chirp)
    phaser._gpios.gpio_burst = 0
    phaser._gpios.gpio_burst = 1
    phaser._gpios.gpio_burst = 0

    time.sleep(0.001)

    # Capture synchronized data
    rx_data = sdr.rx()

    return rx_data

def process_data(raw_data):
    """
    Process raw data
    This is where we can do monopulse
    raw_data is a list of two channels: [ch0, ch1]
    """
    rx_ch = raw_data[0] + raw_data[1]

    return rx_ch

    
# Live beat frequency and range display with spectrogram
print("Starting live FMCW capture...")
print("Press Ctrl+C (Interrupt kernel) to stop")
print()
print(f"Expected beat frequency for 2m target: ~13.3 kHz")
print(f"Range resolution: {3e8/(2*config.chirp_BW):.2f} m")
print()

actual_sample_rate = sdr.sample_rate
chirp_samples = int(actual_sample_rate * config.ramp_time * 1e-6)

print(f"Sample rate: {actual_sample_rate/1e6:.2f} MHz")
print(f"Buffer size: {config.sdr_buf_size} samples")
print(f"Chirp samples: {chirp_samples}")
print()

# Spectrogram history buffer
HISTORY_LENGTH = 100  # Keep last 100 frames
spectrogram_history = []

# Create figure and axes ONCE
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 14))

# Initialize plot elements that will be updated
line1_i, = ax1.plot([], [], 'b-', linewidth=0.8, alpha=0.6, label='I')
line1_q, = ax1.plot([], [], 'r-', linewidth=0.8, alpha=0.6, label='Q')
line1_mag, = ax1.plot([], [], 'g-', linewidth=1.2, label='Mag')
line2_fft, = ax2.plot([], [], 'b-', linewidth=1)
vline1 = None
vspan1 = None
im3 = None
cbar3 = None

ax1.set_xlabel('Time (ms)', fontsize=11)
ax1.set_ylabel('Amplitude', fontsize=11)
ax1.grid(True, alpha=0.3)
ax1.legend(loc='upper right', fontsize=9)

ax2.set_xlabel('Beat Frequency (kHz)', fontsize=11)
ax2.set_ylabel('Magnitude (dB)', fontsize=11)
ax2.grid(True, alpha=0.3)
ax2.set_xlim([60, 140])

ax3.set_xlabel('Beat Frequency (kHz)', fontsize=11)
ax3.set_ylabel('Frame Number', fontsize=11)
ax3.grid(True, alpha=0.3, color='white', linewidth=0.5)

plt.tight_layout()

try:
    frame_count = 0

    while True:
        frame_count += 1

        raw_data = capture_data()
        rx_data = process_data(raw_data)

        # Only process the ramp time...
        rx_chirp = rx_data[:chirp_samples]

        # === Update Plot 1: Time Domain ===
        t_ms = np.arange(len(rx_data)) / actual_sample_rate * 1e3
        i_signal = np.real(rx_data)
        q_signal = np.imag(rx_data)
        magnitude = np.abs(rx_data)

        line1_i.set_data(t_ms, i_signal)
        line1_q.set_data(t_ms, q_signal)
        line1_mag.set_data(t_ms, magnitude)
        
        ax1.set_xlim([0, t_ms[-1]])
        ax1.set_ylim([np.min(i_signal)*1.1, np.max(magnitude)*1.1])
        ax1.set_title(f'De-chirped Signal - Frame {frame_count}', fontsize=13, fontweight='bold')
        
        # Update chirp end marker
        if vline1:
            vline1.remove()
        if vspan1:
            vspan1.remove()
        chirp_end_time = config.ramp_time / 1e3
        vline1 = ax1.axvline(chirp_end_time, color='orange', linestyle='--', linewidth=2, alpha=0.7)
        vspan1 = ax1.axvspan(chirp_end_time, t_ms[-1], alpha=0.2, color='red')

        # === FFT Processing ===
        win = np.blackman(len(rx_chirp))
        windowed = rx_chirp * win
        spectrum = np.fft.fft(windowed, n=config.fft_size)
        spectrum_pos = spectrum[:config.fft_size//2]
        freqs_pos = np.fft.fftfreq(config.fft_size, 1/actual_sample_rate)[:config.fft_size//2]
        magnitude_db = 20 * np.log10(np.abs(spectrum_pos) + 1e-12)

        chirp_time_s = config.ramp_time * 1e-6
        chirp_rate = config.chirp_BW / chirp_time_s
        IF_freq = config.signal_freq
        beat_freq = np.abs(freqs_pos - IF_freq)
        range_bins_m = (3e8 * beat_freq) / (2 * chirp_rate)

        spectrogram_history.append(magnitude_db.copy())
        if len(spectrogram_history) > HISTORY_LENGTH:
            spectrogram_history.pop(0)

        # === Update Plot 2: FFT ===
        line2_fft.set_data(freqs_pos / 1e3, magnitude_db)
        ax2.set_ylim([np.max(magnitude_db)-60, np.max(magnitude_db)+5])
        ax2.set_title(f'FFT Spectrum (Chirp: {chirp_samples} samples)', fontsize=13, fontweight='bold')

        # Clear old peak annotations and redraw
        for txt in ax2.texts:
            txt.remove()
        for artist in ax2.artists:
            artist.remove()
        for line in ax2.lines[1:]:  # Keep first line (FFT), remove peak markers
            line.remove()

        threshold = np.max(magnitude_db) - 15
        peak_indices = []
        for i in range(10, len(magnitude_db)-10):
            if magnitude_db[i] > threshold:
                if magnitude_db[i] > magnitude_db[i-1] and magnitude_db[i] > magnitude_db[i+1]:
                    peak_indices.append(i)

        for i, peak_idx in enumerate(peak_indices[:3]):
            peak_freq = freqs_pos[peak_idx] / 1e3
            peak_range = range_bins_m[peak_idx]
            peak_mag = magnitude_db[peak_idx]
            ax2.plot(peak_freq, peak_mag, 'ro', markersize=10)
            ax2.annotate(f'{peak_freq:.1f} kHz\n{peak_range:.2f} m',
                        xy=(peak_freq, peak_mag), xytext=(10, -20),
                        textcoords='offset points', fontsize=9, ha='left',
                        bbox=dict(boxstyle='round,pad=0.4', fc='yellow', alpha=0.7),
                        arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0.2'))

        # === Update Plot 3: Spectrogram ===
        if len(spectrogram_history) > 1:
            spectrogram_data = np.array(spectrogram_history)
            freq_min_idx = np.argmin(np.abs(freqs_pos / 1e3 - 60))
            freq_max_idx = np.argmin(np.abs(freqs_pos / 1e3 - 140))
            spectrogram_subset = spectrogram_data[:, freq_min_idx:freq_max_idx]
            freqs_subset = freqs_pos[freq_min_idx:freq_max_idx] / 1e3

            if im3 is None:
                # First time: create image and colorbar with constrained width
                extent = [freqs_subset[0], freqs_subset[-1], 0, len(spectrogram_history)]
                im3 = ax3.imshow(spectrogram_subset, aspect='auto', origin='lower',
                               extent=extent, cmap='viridis', interpolation='bilinear',
                               vmin=np.median(spectrogram_subset),
                               vmax=np.max(spectrogram_subset))
                # Create colorbar with specific width so it doesn't shrink axes
                from mpl_toolkits.axes_grid1 import make_axes_locatable
                divider = make_axes_locatable(ax3)
                cax = divider.append_axes("right", size="2%", pad=0.1)
                cbar3 = plt.colorbar(im3, cax=cax)
                cbar3.set_label('Magnitude (dB)', fontsize=10)
            else:
                # Update existing image
                extent = [freqs_subset[0], freqs_subset[-1], 0, len(spectrogram_history)]
                im3.set_data(spectrogram_subset)
                im3.set_extent(extent)
                im3.set_clim(vmin=np.median(spectrogram_subset), vmax=np.max(spectrogram_subset))

            ax3.set_title(f'Range-Time Spectrogram (Last {len(spectrogram_history)} frames)',
                         fontsize=13, fontweight='bold')

        fig.canvas.draw()
        fig.canvas.flush_events()
        clear_output(wait=True)
        display(fig)
        plt.pause(0.01)

except KeyboardInterrupt:
    print("\n\n✓ Live capture stopped")
    print(f"  Total frames captured: {frame_count}")

plt.close()